<a href="https://colab.research.google.com/github/Tritiksha20/Machine-Learning-College/blob/main/13_HIDDEN_MARKOV_MODEL_(HMM)_FOR_SEQUENTIAL_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install hmmlearn

In [7]:
import numpy as np
import pandas as pd
from hmmlearn import hmm
from sklearn.metrics import accuracy_score

# ---- Load dataset ----
df = pd.read_csv("/content/weather_activity_sequence.csv")

activity_map = {'Walk': 0, 'Shop': 1, 'Clean': 2}
weather_map = {'Rainy': 0, 'Sunny': 1}

obs_sequence = df['activity'].map(activity_map).values.reshape(-1, 1)
true_hidden = df['true_weather'].map(weather_map).values

print("Dataset preview:")
print(df.head())
print(f"\nTotal observations (days): {len(obs_sequence)}")


Dataset preview:
   day activity true_weather
0    1    Clean        Rainy
1    2     Walk        Sunny
2    3     Shop        Rainy
3    4    Clean        Rainy
4    5    Clean        Rainy

Total observations (days): 200


In [8]:
# =========================================================
# Train/test split by contiguous time blocks (sequential data
# should not be split randomly)
# =========================================================
split = int(0.8 * len(obs_sequence))
train_obs = obs_sequence[:split]
test_obs = obs_sequence[split:]
test_true_hidden = true_hidden[split:]

In [9]:
# =========================================================
# Fit a Categorical/Multinomial HMM
# =========================================================
model = hmm.CategoricalHMM(n_components=2, n_iter=200, random_state=42)
model.fit(train_obs)

print("\n" + "=" * 60)
print("LEARNED HMM PARAMETERS")
print("=" * 60)
print("Start probabilities:")
print(np.round(model.startprob_, 3))
print("\nTransition matrix (rows=from state, cols=to state):")
print(np.round(model.transmat_, 3))
print("\nEmission probabilities (rows=state, cols=Walk/Shop/Clean):")
print(np.round(model.emissionprob_, 3))


LEARNED HMM PARAMETERS
Start probabilities:
[0.999 0.001]

Transition matrix (rows=from state, cols=to state):
[[0.852 0.148]
 [0.999 0.001]]

Emission probabilities (rows=state, cols=Walk/Shop/Clean):
[[0.237 0.428 0.336]
 [0.729 0.115 0.157]]


In [10]:
# =========================================================
# Decode hidden states for the test sequence (Viterbi algorithm)
# =========================================================
log_prob, hidden_states_pred = model.decode(test_obs, algorithm="viterbi")
print("\n" + "=" * 60)
print("DECODING (Viterbi) ON TEST SEQUENCE")
print("=" * 60)
print(f"Log-probability of the decoded sequence: {log_prob:.2f}")
print(f"Predicted hidden states (first 20): {hidden_states_pred[:20]}")
print(f"Actual hidden states   (first 20): {test_true_hidden[:20]}")

# Note: HMM states are unsupervised/unordered, so state labels (0/1) may
# be swapped relative to the true Rainy/Sunny labels. We align them by
# checking both possible label mappings and picking the better match.
acc_direct = accuracy_score(test_true_hidden, hidden_states_pred)
acc_swapped = accuracy_score(test_true_hidden, 1 - hidden_states_pred)
best_acc = max(acc_direct, acc_swapped)
print(f"\nDecoding accuracy vs true weather (best label alignment): {best_acc:.4f}")


DECODING (Viterbi) ON TEST SEQUENCE
Log-probability of the decoded sequence: -51.57
Predicted hidden states (first 20): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Actual hidden states   (first 20): [0 0 1 1 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1]

Decoding accuracy vs true weather (best label alignment): 0.6000


In [11]:
# =========================================================
# Predict / sample future observations
# =========================================================
n_future_days = 5
sampled_obs, sampled_states = model.sample(n_future_days)
inv_activity_map = {v: k for k, v in activity_map.items()}
predicted_activities = [inv_activity_map[o[0]] for o in sampled_obs]

print("\n" + "=" * 60)
print(f"PREDICTED ACTIVITY SEQUENCE FOR NEXT {n_future_days} DAYS")
print("=" * 60)
print(predicted_activities)


PREDICTED ACTIVITY SEQUENCE FOR NEXT 5 DAYS
['Clean', 'Clean', 'Clean', 'Walk', 'Walk']
